# Calculate Chessmetrics Ratings

In [1]:
import os

# Move up one level to set the working directory to the repo root
os.chdir(os.path.abspath(os.path.join(os.getcwd(), "..")))

In [2]:
# Imports
import numpy as np
import pandas as pd
from scipy.stats import linregress
from tqdm import tqdm
import matplotlib.pyplot as plt

In [3]:
from pathlib import Path

KAGGLE_DATA_PATH = Path(os.path.abspath(os.path.join(os.getcwd(), r"data\kaggle")))
DATA_PATH = Path(os.path.abspath(os.path.join(os.getcwd(), r"data")))

## Functions

In [8]:
def calculate_chessmetrics(
    teams, data, initial_rating=2000, k=140, decay=0.97, importance_weight=1.0
):
    """
    Calculate Chessmetrics ratings for each team based on match data.

    Parameters:
    - teams (array-like): Containing Team-IDs.
    - data (pd.DataFrame): DataFrame with all matches in chronological order.
    - initial_rating (float): Initial rating of an unranked team (default: 2000).
    - k (float): Adjustment factor determining impact of each match on ratings (default: 140).
    - decay (float): Decay factor applied for older matches (default: 0.97).
    - importance_weight (float): Importance scaling factor for tournament matches.

    Returns:
    - list: Historical ratings of the winning team (WTeam).
    - list: Historical ratings of the losing team (LTeam).
    """

    # Dictionary to keep track of team ratings
    team_dict = {team: initial_rating for team in teams}

    # Lists to store ratings
    r1, r2, loss = [], [], []

    # Iterate over games in chronological order
    for season, wteam, lteam, ws, ls, w, tourney in tqdm(
        zip(
            data.Season,
            data.WTeamID,
            data.LTeamID,
            data.WScore,
            data.LScore,
            data.weight,
            data.tourney,
        ),
        total=len(data),
    ):
        # Apply importance weighting for tournament games
        weight = w * (importance_weight if tourney == 1 else 1.0)

        # Ensure a minimum rating (e.g., 100) to prevent zero errors
        team_dict[wteam] = max(team_dict[wteam], 100)
        team_dict[lteam] = max(team_dict[lteam], 100)

        # Compute expected score using Chessmetrics formula (weighted average past performance)
        expected_w = team_dict[wteam] / (team_dict[wteam] + team_dict[lteam])
        expected_l = team_dict[lteam] / (team_dict[wteam] + team_dict[lteam])

        # Update ratings based on performance
        team_dict[wteam] += weight * k * (1 - expected_w)
        team_dict[lteam] += weight * k * (0 - expected_l)

        # Apply **Chessmetrics Decay**: Older matches have less impact
        for team in team_dict:
            team_dict[team] *= decay

        # Store ratings
        r1.append(team_dict[wteam])
        r2.append(team_dict[lteam])
        loss.append((1 - expected_w) ** 2)

    return r1, r2, loss


def create_chessmetrics_data(
    teams, data, initial_rating=2000, k=140, decay=0.97, importance_weight=1.0
):
    """
    Create a DataFrame with Chessmetrics ratings for teams.

    Parameters:
    - teams (array-like): Containing Team-IDs.
    - data (pd.DataFrame): DataFrame with all matches in chronological order.
    - initial_rating (float): Initial rating of an unranked team (default: 2000).
    - k (float): Adjustment factor determining impact of each match on ratings (default: 140).
    - decay (float): Decay factor applied for older matches (default: 0.97).
    - importance_weight (float): Importance scaling factor for tournament matches.

    Returns:
    - DataFrame: Summary statistics of Chessmetrics ratings for teams.
    """

    # Compute Chessmetrics ratings
    r1, r2, loss = calculate_chessmetrics(
        teams, data, initial_rating, k, decay, importance_weight
    )

    # Compute loss only for tournament games
    full_loss = np.mean(np.array(loss)[data.tourney == 1])
    print(f"Loss: {full_loss}")

    # Add a stage1 column to the data
    stage1_seasons = data.Season.max() - 4
    data["stage1"] = (data.tourney == 1) & (data.Season >= stage1_seasons)

    # Compute loss for Stage1 games
    stage1_loss = np.mean(np.array(loss)[data.stage1])
    print(f"Stage1 Loss: {stage1_loss}")

    # Prepare DataFrame for results
    seasons = np.concatenate([data.Season, data.Season])
    days = np.concatenate([data.DayNum, data.DayNum])
    teams = np.concatenate([data.WTeamID, data.LTeamID])
    tourney = np.concatenate([data.tourney, data.tourney])
    ratings = np.concatenate([r1, r2])

    rating_df = pd.DataFrame(
        {
            "Season": seasons,
            "DayNum": days,
            "TeamID": teams,
            "Rating": ratings,
            "Tourney": tourney,
        }
    )

    # Sort DataFrame and compute summary statistics
    rating_df.sort_values(["TeamID", "Season", "DayNum"], inplace=True)
    rating_df = rating_df[rating_df["Tourney"] == 0]
    grouped = rating_df.groupby(["TeamID", "Season"])

    results = grouped["Rating"].agg(["mean", "median", "std", "min", "max", "last"])
    results.columns = [
        "Rating_Mean",
        "Rating_Median",
        "Rating_Std",
        "Rating_Min",
        "Rating_Max",
        "Rating_Last",
    ]
    results["Rating_Trend"] = grouped.apply(
        lambda x: linregress(range(len(x)), x["Rating"]).slope, include_groups=False
    )
    results.reset_index(inplace=True)

    return results

## Apply Functions and Save Results to CSV

In [9]:
# Load and Process Data Men's Tourney
regular_m = pd.read_csv(KAGGLE_DATA_PATH / "MRegularSeasonCompactResults.csv")
tourney_m = pd.read_csv(KAGGLE_DATA_PATH / "MNCAATourneyCompactResults.csv")
teams_m = pd.read_csv(KAGGLE_DATA_PATH / "MTeams.csv")

regular_m["tourney"] = 0
tourney_m["tourney"] = 1
regular_m["weight"] = 1
tourney_m["weight"] = 0.7

data_m = pd.concat([regular_m, tourney_m])
data_m.sort_values(["Season", "DayNum"], inplace=True)
data_m.reset_index(inplace=True, drop=True)

chessmetrics_df_men = create_chessmetrics_data(
    teams_m.TeamID,
    data_m,
    initial_rating=1200,
    k=125,
    decay=0.97,
    importance_weight=1.0,
)
chessmetrics_df_men.tail(10)

100%|██████████| 195015/195015 [00:05<00:00, 35093.68it/s]


Loss: 0.2504084980618035
Stage1 Loss: 0.24926893966666772


,TeamID,Season,Rating_Mean,Rating_Median,Rating_Std,Rating_Min,Rating_Max,Rating_Last,Rating_Trend
13378,1476,2023,88.916667,36.375,61.110841,36.375,157.625,36.375000,1.645439
13379,1476,2024,48.500000,36.375,36.996840,36.375,157.625,36.375000,0.350667
13380,1476,2025,90.728448,36.375,61.367072,36.375,157.625,157.625000,0.358374
13381,1477,2023,81.843750,36.375,59.639162,36.375,157.625,36.375000,0.355572
13382,1477,2024,77.433082,36.375,57.778684,36.375,157.625,55.617468,1.472866
13383,1477,2025,55.931452,36.375,45.332738,36.375,157.625,157.625000,0.880040
13384,1478,2024,86.547414,36.375,60.774139,36.375,157.625,36.375000,2.389163
13385,1478,2025,65.642241,36.375,52.803668,36.375,157.625,36.375000,0.119458
13386,1479,2025,88.339286,36.375,61.104256,36.375,157.625,157.625000,1.924603
13387,1480,2025,56.583333,36.375,45.959694,36.375,157.625,157.625000,1.321746


In [11]:
# Load and Process Data Women's Tourney
regular_w = pd.read_csv(KAGGLE_DATA_PATH / "WRegularSeasonCompactResults.csv")
tourney_w = pd.read_csv(KAGGLE_DATA_PATH / "WNCAATourneyCompactResults.csv")
teams_w = pd.read_csv(KAGGLE_DATA_PATH / "WTeams.csv")

regular_w["tourney"] = 0
tourney_w["tourney"] = 1
regular_w["weight"] = 0.95
tourney_w["weight"] = 1

data_w = pd.concat([regular_w, tourney_w])
data_w.sort_values(["Season", "DayNum"], inplace=True)
data_w.reset_index(inplace=True, drop=True)

chessmetrics_df_women = create_chessmetrics_data(
    teams_w.TeamID,
    data_w,
    initial_rating=1250,
    k=190,
    decay=0.97,
    importance_weight=1.0,
)
chessmetrics_df_women.tail(10)

100%|██████████| 138278/138278 [00:03<00:00, 35453.13it/s]


Loss: 0.256880035963205
Stage1 Loss: 0.2615767416533388


,TeamID,Season,Rating_Mean,Rating_Median,Rating_Std,Rating_Min,Rating_Max,Rating_Last,Rating_Trend
9478,3476,2023,59.481786,9.4575,80.546752,9.4575,184.542500,9.457500,1.533311
9479,3476,2024,32.802167,9.4575,60.534922,9.4575,184.542500,9.457500,1.324336
9480,3476,2025,93.757685,9.4575,89.148918,9.4575,184.542500,184.542500,3.420464
9481,3477,2023,80.757833,9.4575,87.582307,9.4575,190.048160,9.457500,2.115137
9482,3477,2024,88.974789,9.4575,88.058673,9.4575,188.179561,35.716818,1.325624
9483,3477,2025,43.127692,9.4575,70.369896,9.4575,184.542500,9.457500,1.257021
9484,3478,2024,107.942812,184.5425,88.245664,9.4575,184.542500,9.457500,5.327000
9485,3478,2025,46.975714,9.4575,73.160221,9.4575,184.542500,9.457500,4.120774
9486,3479,2025,55.131848,9.4575,78.609271,9.4575,184.542500,9.457500,3.806196
9487,3480,2025,80.788426,9.4575,87.667116,9.4575,184.542500,184.542500,1.817121


In [ ]:
# Save to csv
elo_df_men.to_csv(DATA_PATH / "mens_elo_rating.csv")
elo_df_women.to_csv(DATA_PATH / "womens_elo_rating.csv")

## Top 20 Teams Based on Latest Data-Update (07.03.2025)

In [ ]:
# Men's Teams
tmp_df_men = pd.merge(elo_df_men, teams_m, on="TeamID", how="left")
tmp_df_men = tmp_df_men[tmp_df_men["Season"] == 2025]
top_men_teams = tmp_df_men.sort_values("Rating_Last", ascending=False)[:20][
    ["TeamName", "Rating_Last", "Rating_Trend"]
]
top_men_teams = top_men_teams.reindex(index=top_men_teams.index[::-1])

# Women's Teams
tmp_df_women = pd.merge(elo_df_women, teams_w, on="TeamID", how="left")
tmp_df_women = tmp_df_women[tmp_df_women["Season"] == 2025]
top_women_teams = tmp_df_women.sort_values("Rating_Last", ascending=False)[:20][
    ["TeamName", "Rating_Last", "Rating_Trend"]
]
top_women_teams = top_women_teams.reindex(index=top_women_teams.index[::-1])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 8))

# Men's Teams
ax1.barh(
    top_men_teams["TeamName"],
    top_men_teams["Rating_Last"],
    color="skyblue",
    label="Rating_Last",
)
ax1.set_title("Top Men's Teams - 2025")
ax1.set_xlabel("Last Rating")
ax1.set_ylabel("TeamName")
ax1.legend()

# Women's Teams
ax2.barh(
    top_women_teams["TeamName"],
    top_women_teams["Rating_Last"],
    color="#3F51B5",
    label="Rating_Last",
)
ax2.set_title("Top Women's Teams - 2025")
ax2.set_xlabel("Last Rating")
ax2.set_ylabel("TeamName")
ax2.legend()

plt.tight_layout()
plt.show()

## Hyperparameter Tuning

### Define the Objective Function

In [ ]:
import optuna


def chessmetrics_objective(trial):
    # Sample hyperparameters
    k = trial.suggest_int("k", 100, 150)  # Impact per game
    decay = trial.suggest_float("decay", 0.90, 1.0)  # Decay rate for older matches
    importance_weight = trial.suggest_float(
        "importance_weight", 0.8, 1.5
    )  # Tournament weighting
    initial_rating = trial.suggest_int("initial_rating", 900, 1250)  # Starting rating

    # Compute Chessmetrics ratings with sampled parameters
    r1, r2, loss = calculate_chessmetrics(
        teams_m.TeamID,
        data_m,
        initial_rating=initial_rating,
        k=k,
        decay=decay,
        importance_weight=importance_weight,
    )

    # Compute loss for tournament games
    loss = np.mean(np.array(loss)[data_m.tourney == 1])

    return loss  # Optuna minimizes this

### Run Hyperparameter Optimization

In [ ]:
# Run the hyperparameter optimization
study = optuna.create_study(direction="minimize")
study.optimize(chessmetrics_objective, n_trials=50)

# Print the best parameters found
print("Best Chessmetrics Parameters:", study.best_params)

### Apply the Best Parameters

In [ ]:
best_params = study.best_params

# Compute Chessmetrics ratings using optimized parameters
chessmetrics_df_best = create_chessmetrics_data(
    teams_m.TeamID,
    data_m,
    initial_rating=best_params["initial_rating"],
    k=best_params["k"],
    decay=best_params["decay"],
    importance_weight=best_params["importance_weight"],
)